In [9]:
from openai import OpenAI
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

# https://github.com/speedyapply/JobSpy
jobs = scrape_jobs(
    site_name=["indeed", "linkedin"],
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=72,
    results_wanted=10, 
)

df = pd.DataFrame(jobs)

output_dir = Path("Job Posting Data")
output_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = output_dir / "filtered_jobs.jsonl"

# We ensure the title contains BOTH "AI" and "Engineer" but not "Intern"
mask = (
    df['title'].str.contains('AI', case=False, na=False)
    & df['title'].str.contains('Engineer', case=False, na=False)
    & ~df['title'].str.contains('Intern', case=False, na=False)
)
matching_jobs = df[mask].copy()

# We drop duplicates based on the combination of 'title' and 'company'
filtered_jobs = matching_jobs.drop_duplicates(subset=['title', 'company']).copy()
filtered_jobs.to_json(jsonl_path, orient='records', lines=True, force_ascii=False)

print(f"Saved filtered jobs to: {jsonl_path.resolve()}")
print(f"Total jobs found: {len(df)}")
print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")

results_to_show = filtered_jobs[['title', 'company', 'job_url']].copy()
description_series = filtered_jobs['description'] if 'description' in filtered_jobs.columns else pd.Series('', index=filtered_jobs.index)
results_to_show['description_chars'] = description_series.fillna('').astype(str).str.len()
results_to_show = results_to_show[['title', 'company', 'description_chars', 'job_url']]
results_to_show['job_url'] = results_to_show['job_url'].apply(
    lambda url: f'<a href=\"{url}\" target=\"_blank\">{url}</a>' if pd.notna(url) else ''
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

Saved filtered jobs to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-introduction/Job Posting Data/filtered_jobs.jsonl
Total jobs found: 20
Jobs matching 'AI' + 'Engineer' in title: 15
Jobs after deduplicating identical title/company pairs: 14


title,company,description_chars,job_url
AI Engineer (DevOps),Wipro,3275,https://www.indeed.com/viewjob?jk=f827c33d4ee013f2
Principal AI Engineer,VideoAmp,6285,https://www.indeed.com/viewjob?jk=a59140bb24e25483
"AI Engineer, Enterprise AI (Principal/Sr. Principal/Distinguished)",Palo Alto Networks,6890,https://www.indeed.com/viewjob?jk=07564be4ba2a6d92
Junior Applied AI Engineer- HRIS,Fortinet,4095,https://www.indeed.com/viewjob?jk=19fc4813887bb16d
Microservices Tech Lead – AI Engineer/Analyst - Vice President – TAMPA,Information Technology Senior Management Forum,4401,https://www.indeed.com/viewjob?jk=911b6e2b9ebbe41b
AI Engineer,SoFi,6669,https://www.linkedin.com/jobs/view/4393576713
Python AI Engineer,Arka Innovate,2497,https://www.linkedin.com/jobs/view/4392455560
Senior AI Engineer,H2O.ai,6754,https://www.linkedin.com/jobs/view/4393242437
AI Engineer,Oteemo Inc.,8330,https://www.linkedin.com/jobs/view/4393208617
Senior AI Engineer,LinkedIn,7072,https://www.linkedin.com/jobs/view/4391882675


In [ ]:
import re

if filtered_jobs.empty:
    raise ValueError("filtered_jobs is empty. Please adjust your filters before summarizing.")

if "description" not in filtered_jobs.columns:
    raise KeyError("filtered_jobs must contain a 'description' column.")

summary_source_jobs = filtered_jobs.copy()
summary_source_jobs = summary_source_jobs[summary_source_jobs["description"].notna()].copy()
summary_source_jobs["description"] = summary_source_jobs["description"].astype(str).str.strip()
summary_source_jobs = summary_source_jobs[summary_source_jobs["description"] != ""].copy()

if "job_url" in summary_source_jobs.columns:
    summary_source_jobs = summary_source_jobs.drop_duplicates(subset=["job_url"])
else:
    dedupe_columns = [
        column
        for column in ["title", "company", "description"]
        if column in summary_source_jobs.columns
    ]
    summary_source_jobs = summary_source_jobs.drop_duplicates(subset=dedupe_columns)

if summary_source_jobs.empty:
    raise ValueError("filtered_jobs contains no usable job descriptions after cleaning.")

def clean_description(text):
    text = str(text)
    text = text.replace("**", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

job_entries = []

for _, job in summary_source_jobs.iterrows():
    description = clean_description(job["description"])
    if not description:
        continue

    title = str(job.get("title", "Unknown title")).strip()
    company = str(job.get("company", "Unknown company")).strip()
    job_url = str(job.get("job_url", "")).strip()

    entry_lines = [
        f"Job {len(job_entries) + 1}",
        f"Title: {title}",
        f"Company: {company}",
        f"Description: {description}",
    ]

    if job_url and job_url.lower() != "nan":
        entry_lines.append(f"URL: {job_url}")

    job_entries.append("\n".join(entry_lines))

if not job_entries:
    raise ValueError("No non-empty job descriptions were available for summarization.")

job_descriptions = "\n\n---\n\n".join(job_entries)

prompt = f"""
You are analyzing scraped job postings for AI Engineer roles.

Read every job description below and summarize the shared responsibilities across the roles.
Ignore company marketing, benefits, equal opportunity text, visa/location boilerplate, and generic recruiting language.
Focus on repeated responsibilities, expected day-to-day work, collaboration patterns, and technical ownership.

Return your answer in this format:
## Core responsibilities
- 8-12 concise bullet points

## Role focus
- One short paragraph

## Notable differences
- 3-5 bullet points for responsibilities that only appear in some postings

## Machine Learning Knowledge Required
- Describe how much knowledge about Machine Leaning (building models from scratch vs. using pre-trained models, etc.) is expected across the job postings

## Use Cases
- Describe the most common use cases mentioned across the job postings

Job descriptions:
{job_descriptions}
""".strip()

messages = [
    {
        "role": "system",
        "content": "You analyze scraped job postings and identify the true recurring responsibilities across them.",
    },
    {"role": "user", "content": prompt},
]

print(f"Summarizing {len(job_entries)} scraped job descriptions")
print(f"Combined description length: {len(job_descriptions):,} characters")
messages

In [ ]:
client = OpenAI()

response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
summary = response.choices[0].message.content
print(summary)